# WEIGHT1 — P0 가중치 조절형 RL

새 Colab 런타임에서 아래 칸을 순서대로 실행합니다. Google Drive의 `MyDrive/FutureTraffic/` 폴더에 `FutureTraffic_WEIGHT1.zip`을 먼저 올려 둡니다.

실험은 **동등성 확인 → 고정 모드 비교 → 짧은 학습 → 통과 시 본학습·확인 평가** 순서로 자동 진행합니다. 초기 확인은 누적 6시간, 전체는 누적 48시간 상한이며 병렬 작업도 합산합니다. 기준 미통과 또는 예산 부족이면 확대를 멈춥니다.

기존 프로젝트 폴더·노트북·ZIP은 보존합니다. 새 코드는 `/content/FutureTraffic_WEIGHT1`, 결과·재개 기록은 Drive의 `explore_weight_v1`에 저장합니다. 기존 날짜를 재사용한 추가 개발 확인 평가이며, 새 날짜나 실제 도로 성능을 입증하는 실험은 아닙니다.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE_ROOT / "FutureTraffic_WEIGHT1.zip"
WORKSPACE = Path("/content/FutureTraffic_WEIGHT1")
RESULTS = DRIVE_ROOT / "explore_weight_v1"
assert ARCHIVE.is_file(), f"WEIGHT1 ZIP을 먼저 업로드하세요: {ARCHIVE}"
print("ZIP:", ARCHIVE)
print("결과 및 재개 기록:", RESULTS)


In [ ]:
# 패키지 파일 전체의 SHA-256을 확인한 뒤, 검증된 전용 추출기를 사용합니다.
# 기존 폴더를 삭제하지 않고 ZIP에 포함된 파일만 교체합니다.
import hashlib
import json
import stat
import sys
import types
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "WEIGHT1 전용 ZIP이 아닙니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("weight1_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE}")


In [ ]:
import os
import subprocess

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)


In [ ]:
# WEIGHT1 필수 검사를 통과해야 실험을 시작합니다.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_weight*.py", "-v"],
               cwd=WORKSPACE, check=True)
subprocess.run([sys.executable, "src/explore_weight.py", "prepare", "--out", str(RESULTS)],
               cwd=WORKSPACE, check=True)


In [ ]:
# 첫 실행과 중단 후 재개 모두 같은 명령입니다. 판정·예산에 따라 다음 단계로 자동 진행합니다.
subprocess.run([sys.executable, "src/explore_weight.py", "run",
                "--out", "/content/drive/MyDrive/FutureTraffic/explore_weight_v1",
                "--work", "/content/weight1_work", "--workers", "2"],
               cwd=WORKSPACE, check=True)


In [ ]:
from IPython.display import Markdown, display

subprocess.run([sys.executable, "src/explore_weight.py", "summary", "--out", str(RESULTS)],
               cwd=WORKSPACE, check=True)
summary_file = RESULTS / "summary.md"
if summary_file.is_file():
    display(Markdown(summary_file.read_text()))
else:
    print("아직 요약 파일이 없습니다. 위 실행 로그를 확인하세요.")
print("원시 평가·모델·예산 장부:", RESULTS)
